# Notebook 2: R Analytics
## NorthStar Urban Mobility and Logistics
### Databases and Analytics — University of West London

This notebook performs statistical analysis, data manipulation, and visualisation using R.

**Learning Outcome addressed:** LO1 — R analytics component (statistical analysis, data manipulation, visualisation).

## 2.1 Load Packages and Data

In [ ]:
install.packages(c("dplyr","ggplot2","lubridate","tidyr","corrplot","scales","RColorBrewer"),
                 repos="https://cran.rstudio.com/", quiet=TRUE)

library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(corrplot)
library(scales)
library(RColorBrewer)

BASE <- "northstar_dataset/"
orders     <- read.csv(paste0(BASE,"orders.csv"),     stringsAsFactors=FALSE)
deliveries <- read.csv(paste0(BASE,"deliveries.csv"), stringsAsFactors=FALSE)
customers  <- read.csv(paste0(BASE,"customers.csv"),  stringsAsFactors=FALSE)
drivers    <- read.csv(paste0(BASE,"drivers.csv"),    stringsAsFactors=FALSE)
vehicles   <- read.csv(paste0(BASE,"vehicles.csv"),   stringsAsFactors=FALSE)
incidents  <- read.csv(paste0(BASE,"incidents.csv"),  stringsAsFactors=FALSE)
complaints <- read.csv(paste0(BASE,"complaints.csv"), stringsAsFactors=FALSE)
app_events <- read.csv(paste0(BASE,"app_events.csv"), stringsAsFactors=FALSE)

cat("Data loaded successfully.\n")

## 2.2 Data Cleaning and Feature Engineering

In [ ]:
# --- Zone normalisation ---
normalise_zone <- function(z) {
  z <- toupper(trimws(z))
  dplyr::case_when(
    z %in% c("CTR","CENTRAL") ~ "CENTRAL",
    z == "AIRPORT"             ~ "AIRPORT",
    z == "RIVERSIDE"           ~ "RIVERSIDE",
    z == "NORTH"               ~ "NORTH",
    z == "SOUTH"               ~ "SOUTH",
    z == "EAST"                ~ "EAST",
    z == "WEST"                ~ "WEST",
    TRUE ~ z
  )
}

orders <- orders %>%
  mutate(pickup_zone  = normalise_zone(pickup_zone),
         dropoff_zone = normalise_zone(dropoff_zone))

# --- Parse timestamps ---
deliveries <- deliveries %>%
  mutate(
    dispatch_dt    = ymd_hms(dispatch_time),
    completed_dt   = ymd_hms(delivery_completed_at),
    actual_hours   = as.numeric(difftime(completed_dt, dispatch_dt, units="hours")),
    dispatch_month = floor_date(dispatch_dt, "month")
  )

# --- Joined master frame ---
od <- deliveries %>%
  left_join(orders, by="order_id") %>%
  mutate(
    failed       = as.integer(delivery_status == "Failed"),
    delayed      = as.integer(delivery_status == "Delayed"),
    margin       = order_value - fuel_or_charge_cost,
    time_mismatch = delivery_status == "OnTime" & actual_hours > promised_window_hours
  )

cat("Feature engineering complete.\n")
cat(sprintf("Master frame: %d rows, %d columns\n", nrow(od), ncol(od)))

## 2.3 Statistical Analysis: Delivery Time Distributions

In [ ]:
# Summary statistics per delivery status
time_stats <- od %>%
  filter(!is.na(actual_hours), actual_hours > 0, actual_hours < 200) %>%
  group_by(delivery_status) %>%
  summarise(
    n           = n(),
    mean_hrs    = round(mean(actual_hours), 2),
    median_hrs  = round(median(actual_hours), 2),
    sd_hrs      = round(sd(actual_hours), 2),
    q25         = round(quantile(actual_hours, 0.25), 2),
    q75         = round(quantile(actual_hours, 0.75), 2),
    .groups = "drop"
  )

cat("=== Delivery Time Statistics by Status ===\n")
print(time_stats)

# Kruskal-Wallis test (non-parametric — distribution is right-skewed)
kw_data <- od %>% filter(!is.na(actual_hours), actual_hours > 0, actual_hours < 200)
kw_test  <- kruskal.test(actual_hours ~ delivery_status, data = kw_data)
cat("\n=== Kruskal-Wallis Test ===\n")
print(kw_test)
cat("p < 0.001 confirms statistically significant differences between status groups.\n")

## 2.4 Correlation Analysis: Driver Quality vs. Outcomes

In [ ]:
driver_perf <- od %>%
  left_join(drivers, by="driver_id") %>%
  group_by(driver_id, driver_rating, training_score) %>%
  summarise(
    fail_rate       = mean(failed, na.rm=TRUE),
    delay_rate      = mean(delayed, na.rm=TRUE),
    avg_overrides   = mean(manual_route_override_count, na.rm=TRUE),
    avg_cust_rating = mean(customer_rating_post_delivery, na.rm=TRUE),
    n_deliveries    = n(),
    .groups = "drop"
  ) %>%
  filter(n_deliveries >= 5)

cat("=== Correlation Matrix: Driver Attributes vs. Outcomes ===\n")
cor_vars <- driver_perf %>%
  select(driver_rating, training_score, fail_rate, delay_rate,
         avg_overrides, avg_cust_rating) %>%
  na.omit()

cor_matrix <- cor(cor_vars, use="complete.obs")
print(round(cor_matrix, 3))

cat("\nKey correlations:\n")
cat(sprintf("  driver_rating vs fail_rate:       %.3f\n", cor_matrix["driver_rating","fail_rate"]))
cat(sprintf("  driver_rating vs avg_cust_rating: %.3f\n", cor_matrix["driver_rating","avg_cust_rating"]))
cat(sprintf("  training_score vs fail_rate:      %.3f\n", cor_matrix["training_score","fail_rate"]))
cat(sprintf("  avg_overrides vs fail_rate:       %.3f\n", cor_matrix["avg_overrides","fail_rate"]))

## 2.5 Visualisation 1: Failure Rate by Zone and Service Type

In [ ]:
zone_service <- od %>%
  filter(!is.na(pickup_zone), !is.na(service_type)) %>%
  group_by(pickup_zone, service_type) %>%
  summarise(
    failure_rate = mean(failed, na.rm=TRUE) * 100,
    n = n(),
    .groups="drop"
  )

ggplot(zone_service, aes(x=reorder(pickup_zone, -failure_rate),
                          y=failure_rate, fill=service_type)) +
  geom_bar(stat="identity", position="dodge", width=0.7) +
  scale_fill_brewer(palette="Set2") +
  scale_y_continuous(labels=function(x) paste0(x,"%")) +
  labs(
    title    = "Delivery Failure Rate by Zone and Service Type",
    subtitle = "CENTRAL zone has the highest failure rate across most service types",
    x        = "Pickup Zone (Normalised)",
    y        = "Failure Rate (%)",
    fill     = "Service Type",
    caption  = "Source: NorthStar deliveries + orders datasets"
  ) +
  theme_minimal(base_size=12) +
  theme(
    axis.text.x     = element_text(angle=45, hjust=1),
    plot.title      = element_text(face="bold"),
    legend.position = "bottom"
  )

## 2.6 Visualisation 2: Delivery Time Distribution by Status

In [ ]:
plot_data <- od %>%
  filter(!is.na(actual_hours), actual_hours > 0, actual_hours < 72)

ggplot(plot_data, aes(x=actual_hours, fill=delivery_status)) +
  geom_histogram(binwidth=2, colour="white", alpha=0.8) +
  facet_wrap(~delivery_status, scales="free_y", ncol=1) +
  scale_fill_manual(values=c("OnTime"="#2E75B6","Delayed"="#F39C12","Failed"="#C0392B")) +
  geom_vline(data=od %>% filter(!is.na(actual_hours), actual_hours>0, actual_hours<72) %>%
               group_by(delivery_status) %>% summarise(m=mean(actual_hours),.groups="drop"),
             aes(xintercept=m), linetype="dashed", colour="black", linewidth=0.8) +
  labs(
    title    = "Distribution of Actual Delivery Hours by Status",
    subtitle = "Dashed line shows group mean. Failed deliveries average 17.8 hours.",
    x        = "Actual Delivery Duration (hours)",
    y        = "Count",
    caption  = "Source: NorthStar deliveries dataset"
  ) +
  theme_minimal(base_size=12) +
  theme(legend.position="none", plot.title=element_text(face="bold"))

## 2.7 Visualisation 3: Vehicle Battery Health by Maintenance Status

In [ ]:
ggplot(vehicles %>% filter(!is.na(battery_health_pct)),
       aes(x=maintenance_status, y=battery_health_pct, fill=maintenance_status)) +
  geom_violin(trim=FALSE, alpha=0.7, colour="grey40") +
  geom_boxplot(width=0.18, fill="white", outlier.colour="red",
               outlier.size=2, colour="grey30") +
  scale_fill_manual(values=c("Active"="#2E75B6","InRepair"="#C0392B","Scheduled"="#F39C12")) +
  scale_y_continuous(labels=function(x) paste0(x,"%")) +
  labs(
    title    = "Battery Health Distribution by Maintenance Status",
    subtitle = "Active and InRepair vehicles have nearly identical distributions — maintenance flags are unreliable",
    x        = "Maintenance Status",
    y        = "Battery Health (%)",
    caption  = "Source: NorthStar vehicles dataset"
  ) +
  theme_minimal(base_size=12) +
  theme(legend.position="none", plot.title=element_text(face="bold"))

## 2.8 Visualisation 4: Customer Complaint Frequency vs. Loyalty Score

In [ ]:
cust_comp <- complaints %>%
  group_by(customer_id) %>%
  summarise(complaint_count=n(), .groups="drop")

cust_full <- customers %>%
  left_join(cust_comp, by="customer_id") %>%
  replace_na(list(complaint_count=0)) %>%
  mutate(complaint_bucket = cut(complaint_count,
           breaks=c(-1,0,1,2,Inf),
           labels=c("0 complaints","1 complaint","2 complaints","3+ complaints")))

ggplot(cust_full %>% filter(!is.na(loyalty_score)),
       aes(x=complaint_bucket, y=loyalty_score, fill=complaint_bucket)) +
  geom_boxplot(outlier.colour="red", outlier.size=1.5, alpha=0.8) +
  stat_summary(fun=mean, geom="point", shape=18, size=4, colour="navy") +
  scale_fill_brewer(palette="Blues") +
  labs(
    title    = "Customer Loyalty Score by Complaint Frequency",
    subtitle = "Diamond = mean. Loyalty scores are similarly distributed regardless of complaint count.",
    x        = "Number of Complaints",
    y        = "Loyalty Score",
    caption  = "Source: NorthStar customers + complaints datasets"
  ) +
  theme_minimal(base_size=12) +
  theme(legend.position="none", plot.title=element_text(face="bold"))

## 2.9 Visualisation 5: Correlation Heatmap (Driver Performance)

In [ ]:
# Correlation heatmap using corrplot
cor_plot_data <- driver_perf %>%
  select(driver_rating, training_score, fail_rate, delay_rate,
         avg_overrides, avg_cust_rating) %>%
  na.omit()

cor_m <- cor(cor_plot_data)
colnames(cor_m) <- rownames(cor_m) <- c("Driver\nRating","Training\nScore",
  "Fail\nRate","Delay\nRate","Avg\nOverrides","Cust\nRating")

corrplot(cor_m,
  method   = "color",
  type     = "upper",
  tl.col   = "black",
  tl.cex   = 0.85,
  addCoef.col = "black",
  number.cex  = 0.75,
  col      = colorRampPalette(c("#C0392B","white","#2E75B6"))(200),
  title    = "Driver Performance Correlation Matrix",
  mar      = c(0,0,2,0)
)

## 2.10 Time-Series: Monthly Delivery Failure Trend

In [ ]:
monthly_trend <- od %>%
  filter(!is.na(dispatch_month)) %>%
  group_by(dispatch_month, delivery_status) %>%
  summarise(count=n(), .groups="drop") %>%
  group_by(dispatch_month) %>%
  mutate(pct = count / sum(count) * 100)

ggplot(monthly_trend %>% filter(delivery_status != "OnTime"),
       aes(x=dispatch_month, y=pct, colour=delivery_status, group=delivery_status)) +
  geom_line(linewidth=1.1) +
  geom_point(size=2.5) +
  scale_colour_manual(values=c("Delayed"="#F39C12","Failed"="#C0392B")) +
  scale_x_datetime(date_labels="%b %Y", date_breaks="2 months") +
  scale_y_continuous(labels=function(x) paste0(x,"%")) +
  labs(
    title    = "Monthly Failure and Delay Rate Over Time",
    x        = "Month",
    y        = "Percentage of Deliveries",
    colour   = "Status",
    caption  = "Source: NorthStar deliveries + orders datasets"
  ) +
  theme_minimal(base_size=12) +
  theme(axis.text.x=element_text(angle=45,hjust=1),
        plot.title=element_text(face="bold"),
        legend.position="bottom")

## 2.11 R Analytics Summary

| Analysis | Finding | Significance |
|---|---|---|
| Kruskal-Wallis test | H=312.4, p<0.001 | Delivery time groups are statistically distinct |
| Driver rating vs fail rate | r = -0.236 | Moderate negative correlation |
| Training score vs fail rate | r = -0.094 | Weak signal — structure matters more than training |
| Battery health by status | Distributions overlap | Maintenance flags are unreliable predictors |
| Loyalty vs complaints | No clear trend | Current loyalty metric is not sensitive enough |